In [ ]:
# =====================
# 0) Imports & config
# =====================
import os, json, glob, math
import numpy as np
import pandas as pd
from functools import reduce
import joblib


In [ ]:
import numpy as np

# --- Paths (edit if needed) ---
MODEL_PATH = "/rhome/zli529/lab/LncRNA_chip_prediction/Final/window1000_bin100/model_training/rf_groupcv_out/rf_groupcv_pipeline.joblib"
PRED_DIR   = "/rhome/zli529/lab/LncRNA_chip_prediction/Final/window1000_bin100/prediction/intergenic_bins"  # directory with the CSVs you listed
OUT_DIR    = "/rhome/zli529/lab/LncRNA_chip_prediction/Final/window1000_bin100/prediction/results_intergenic_TSS"

In [ ]:
# Create output dir
os.makedirs(OUT_DIR, exist_ok=True)

In [ ]:
# Keys shared by all files
KEY_COLS = ["chr","mid","ID","strand"]


INCLUDE_FILES = [
    "ATAC_intergenic_bins_pm1000bp_20bins.csv",
    "H2A.Zac_intergenic_bins_pm1000bp_20bins.csv",
    "H2A.Z_intergenic_bins_pm1000bp_20bins.csv",
    "H2B.Z_intergenic_bins_pm1000bp_20bins.csv",
    "H3_intergenic_bins_pm1000bp_20bins.csv",
    "H3K18ac_intergenic_bins_pm1000bp_20bins.csv",
    "H3K18me_intergenic_bins_pm1000bp_20bins.csv",
    "H3K27ac_intergenic_bins_pm1000bp_20bins.csv",
    "H3K27me_intergenic_bins_pm1000bp_20bins.csv",
    "H3K36me3_intergenic_bins_pm1000bp_20bins.csv",
    "H3K4me1_intergenic_bins_pm1000bp_20bins.csv",
    "H3K4me2_intergenic_bins_pm1000bp_20bins.csv",
    "H3K4me3_intergenic_bins_pm1000bp_20bins.csv",
    "H3K4me_intergenic_bins_pm1000bp_20bins.csv",
    "H3K9ac_intergenic_bins_pm1000bp_20bins.csv",
    "H3R17me2_intergenic_bins_pm1000bp_20bins.csv",
    "MNase_intergenic_bins_pm1000bp_20bins.csv",
]

In [ ]:
BIN_WIDTH = 100  # each 'mid' represents a ~100bp bin center
POS_CLASS = 1    # positive class label the model learned for gene starts
THRESHOLD = 0.7  # default classification threshold for calling positives
MIN_PROB_FOR_PEAK = 0.7   # minimum probability to consider a local peak
PEAK_NEIGHBORHOOD = 3     # number of bins on each side to require as lower (local maxima)
MIN_PEAK_DISTANCE_BP = 200  # prune peaks closer than this (greedy by score)

In [ ]:
# 1) Load & merge intergenic CSVs
# ==================================
def load_one(path):
    df = pd.read_csv(path)
    # Basic sanity
    missing_keys = [c for c in KEY_COLS if c not in df.columns]
    if missing_keys:
        raise ValueError(f"{os.path.basename(path)} is missing key columns: {missing_keys}")
    # Ensure numeric on feature columns (keep key cols as-is)
    for c in df.columns:
        if c not in KEY_COLS:
            df[c] = pd.to_numeric(df[c], errors='coerce')
    return df

frames = []
for fname in INCLUDE_FILES:
    fpath = os.path.join(PRED_DIR, fname)
    if not os.path.exists(fpath):
        print(f"[WARN] Missing file; skipping: {fname}")
        continue
    print(f"Reading {fname} ...")
    frames.append(load_one(fpath))

if not frames:
    raise RuntimeError("No input files were found. Check PRED_DIR/INCLUDE_FILES.")

# Merge on keys
print("Merging feature tables ...")
merged = reduce(lambda l, r: pd.merge(l, r, on=KEY_COLS, how='outer'), frames)

# Fill NaNs with 0 (no signal)
merged = merged.fillna(0)

# Deduplicate any accidentally repeated columns (keep first)
merged = merged.loc[:, ~merged.columns.duplicated(keep='first')]

print(merged.shape)
merged.head()

In [ ]:
# ==================================
# 2) Load trained pipeline
# ==================================
pipe = joblib.load(MODEL_PATH)
print(pipe)



In [ ]:
# ==================================
# 3) Build feature matrix X in the exact order expected by the pipeline
# ==================================
def infer_expected_feature_order(pipeline, candidate_cols):

    # Try: check steps for feature_names_in_
    for name, step in getattr(pipeline, "named_steps", {}).items():
        if hasattr(step, "feature_names_in_"):
            return list(step.feature_names_in_)
    # Try pipeline itself
    if hasattr(pipeline, "feature_names_in_"):
        return list(pipeline.feature_names_in_)
    # Otherwise, use candidate columns
    return list(candidate_cols)

# Candidate feature columns are all numeric non-key columns
numeric_cols = [c for c in merged.columns if c not in KEY_COLS]
expected_cols = infer_expected_feature_order(pipe, numeric_cols)

# Reindex to expected order; fill missing with NaN (imputer in pipeline will handle)
X = merged.reindex(columns=expected_cols)
print(f"X shape: {X.shape}; columns aligned to expected order ({len(expected_cols)} features)")

In [ ]:
# 1) Build a map from expected training names -> available column names
def candidate_sources_for(expected_name, cols):
    cand = []
    # exact
    if expected_name in cols: 
        cand.append(expected_name)
    # drop the "_TSS" token (training-time convention)
    no_tss = expected_name.replace("_TSS", "")
    if no_tss in cols:
        cand.append(no_tss)
    # a few common normalizations
    # e.g. H2B.Z vs H2B.Zac, dashes vs dots, etc. — add rules as needed
    cand = [c for c in cand if c in cols]
    return cand

# 2) Create an aligned frame with exactly the expected columns
feature_cols_available = [c for c in merged.columns if c not in ["chr","mid","ID","strand"]]
cols_set = set(feature_cols_available)

aligned = merged[["chr","mid","ID","strand"]].copy()
for col in expected_cols:
    # choose the first source that exists
    sources = candidate_sources_for(col, cols_set)
    if len(sources) == 0:
        # if we truly have nothing, initialize a zero column (no signal)
        aligned[col] = 0.0
    else:
        aligned[col] = merged[sources[0]]

# 3) Make sure everything is numeric and NaNs → 0
for c in expected_cols:
    aligned[c] = pd.to_numeric(aligned[c], errors="coerce")

aligned[expected_cols] = aligned[expected_cols].fillna(0.0)

# 4) Sanity checks
print("NaNs remaining in aligned features:", aligned[expected_cols].isna().sum().sum())
print("Nonzero entries:", (aligned[expected_cols] != 0).sum().sum())
print("Example row:\n", aligned[expected_cols].iloc[0, :20])

# 5) Finally, build X from the aligned matrix
X = aligned[expected_cols]

In [ ]:
# ==================================
# 4) Predict probabilities & labels
# ==================================
# Some pipeline estimators don't expose predict_proba (e.g., SVM with no probability).
# For RandomForestClassifier it does; but we guard just in case.
if hasattr(pipe, "predict_proba"):
    proba = pipe.predict_proba(X)  # shape (n, 2) for binary
    # Find the column index for the positive class POS_CLASS
    classes_ = pipe.classes_ if hasattr(pipe, "classes_") else np.array([0,1])
    if POS_CLASS in classes_:
        pos_idx = list(classes_).index(POS_CLASS)
    else:
        # assume 1 is positive
        pos_idx = 1 if proba.shape[1] > 1 else 0
    scores = proba[:, pos_idx]
else:
    # Fallback: use decision_function or raw predict (not ideal for calibrated probs)
    if hasattr(pipe, "decision_function"):
        df_vals = pipe.decision_function(X)
        # scale to 0..1 via logistic-ish transform for convenience
        scores = 1/(1+np.exp(-df_vals.astype(float)))
    else:
        preds = pipe.predict(X).astype(int)
        scores = preds.astype(float)

preds = (scores >= THRESHOLD).astype(int)

pred_df = merged[KEY_COLS].copy()
pred_df["score"] = scores
pred_df["pred_label"] = preds

out_pred_csv = os.path.join(OUT_DIR, "predictions_all_0.7_0.7.csv")
pred_df.to_csv(out_pred_csv, index=False)
print(f"Saved predictions -> {out_pred_csv}")
pred_df.head()

In [ ]:
# ==================================
# 5) Export bedGraph for genome browsers
# ==================================
# We map each 'mid' (bin center) to a 100bp interval: [mid-50, mid+50)
start = (pred_df["mid"].astype(int) - (BIN_WIDTH//2)).clip(lower=0)
end   = start + BIN_WIDTH

bedgraph = pd.DataFrame({
    "chr": pred_df["chr"],
    "start": start,
    "end": end,
    "score": pred_df["score"].astype(float)
})

# Sort per bedGraph spec
chrom_ordered = bedgraph.sort_values(["chr","start","end"]).reset_index(drop=True)
bg_path = os.path.join(OUT_DIR, "predictions_0.7_0.7.bedgraph")
chrom_ordered.to_csv(bg_path, sep="\t", header=False, index=False)
print(f"Saved bedGraph -> {bg_path}")
chrom_ordered.head()

In [ ]:
# ==================================
# 6) Simple local-peak calling to propose TSS candidates
# ==================================
def local_maxima_1d(arr, neighborhood=3):
    n = len(arr)
    if n == 0:
        return np.zeros(0, dtype=bool)
    peaks = np.ones(n, dtype=bool)
    for k in range(1, neighborhood+1):
        left  = np.r_[arr[k:], np.full(k, -np.inf)]
        right = np.r_[np.full(k, -np.inf), arr[:-k]]
        peaks &= (arr > left) & (arr > right)
    return peaks

def prune_by_distance(df, min_dist_bp=300):
    keep = []
    last_pos_per_cs = {}  # (chr,strand) -> last_mid_kept
    for _, row in df.sort_values(["score"], ascending=False).iterrows():
        key = (row["chr"], row["strand"])
        last = last_pos_per_cs.get(key, None)
        if last is None or abs(int(row["mid"]) - int(last)) >= min_dist_bp:
            keep.append(row)
            last_pos_per_cs[key] = int(row["mid"])
    return pd.DataFrame(keep)

# Group by chromosome & strand when calling local maxima
cand_list = []
for (chrom, strand), grp in pred_df.groupby(["chr","strand"], sort=False):
    a = grp["score"].values.astype(float)
    peaks_mask = local_maxima_1d(a, neighborhood=PEAK_NEIGHBORHOOD)
    peaks_df = grp.loc[peaks_mask].copy()
    peaks_df = peaks_df[peaks_df["score"] >= MIN_PROB_FOR_PEAK]
    cand_list.append(peaks_df)

cands = pd.concat(cand_list, ignore_index=True) if cand_list else pd.DataFrame(columns=list(pred_df.columns))
print(f"Raw local peaks: {len(cands)}")

# Prune peaks too close to each other (greedy by score)
cands_pruned = prune_by_distance(cands, min_dist_bp=MIN_PEAK_DISTANCE_BP)
print(f"Peaks after distance pruning: {len(cands_pruned)}")

In [ ]:
# Save as BED
tss_bed = pd.DataFrame({
    "chr": cands_pruned["chr"],
    "start": (cands_pruned["mid"].astype(int) - (BIN_WIDTH//2)).clip(lower=0),
    "end":   (cands_pruned["mid"].astype(int) + (BIN_WIDTH//2)),
    "name":  cands_pruned["ID"].astype(str),
    "score": (cands_pruned["score"]*1000).round().astype(int),  # UCSC-style integer score
    "strand": cands_pruned["strand"]
}).sort_values(["chr","start","end"])

tss_bed_path = os.path.join(OUT_DIR, "tss_candidates.bed")
tss_bed.to_csv(tss_bed_path, sep="\\t", header=False, index=False)
print(f"Saved TSS candidate peaks -> {tss_bed_path}")
tss_bed.head()

In [ ]:
# ==================================
# 7) Small QC: score distribution & top sites
# (Optional plotting; uncomment if you want quick looks)
# ==================================
import matplotlib.pyplot as plt
plt.hist(pred_df['score'].values, bins=100)
plt.xlabel("Predicted probability (gene start)")
plt.ylabel("Count")
plt.title("Intergenic site scores")
plt.show()

# Show top 10 by score
pred_df.sort_values("score", ascending=False).head(10)

In [ ]:
KEY_COLS = ["chr","mid","ID","strand"]

# 1) Choose source column for each expected feature
def choose_source(expected, cols):
    if expected in cols:                      # exact match
        return expected
    no_tss = expected.replace("_TSS", "")     # training name vs file name
    if no_tss in cols:
        return no_tss
    return None

feature_cols_avail = [c for c in merged.columns if c not in KEY_COLS]
cols_set = set(feature_cols_avail)
src_map = {col: choose_source(col, cols_set) for col in expected_cols}

# 2) Assemble X in one concat (fast), fill missing with 0, cast to float32
series_list, names = [], []
for col in expected_cols:
    src = src_map[col]
    if src is None:
        s = pd.Series(0.0, index=merged.index)
    else:
        s = pd.to_numeric(merged[src], errors="coerce").fillna(0.0)
    series_list.append(s.astype("float32"))
    names.append(col)

X = pd.concat(series_list, axis=1)
X.columns = names
X = X.copy()  # de-fragment
# Keep keys
keys_df = merged[KEY_COLS].reset_index(drop=True)

# ---- Sanity checks on X (this is what matters) ----
print("NaNs in X:", X.isna().sum().sum())
print("Nonzero entries in X:", (X != 0).sum().sum())
print("X shape:", X.shape)
print("First 10 cols of first row:\n", X.iloc[0, :10].to_string())

In [ ]:
# Use the SAME 'X' you just built (shape 191342 x 153) and your loaded 'pipe'
if hasattr(pipe, "predict_proba"):
    proba = pipe.predict_proba(X)
    classes_ = pipe.classes_ if hasattr(pipe, "classes_") else np.array([0,1])
    pos_idx = list(classes_).index(1) if 1 in classes_ else (1 if proba.shape[1] > 1 else 0)
    scores = proba[:, pos_idx].astype("float32")
else:
    scores = (1/(1+np.exp(-pipe.decision_function(X).astype(float)))).astype("float32") \
             if hasattr(pipe, "decision_function") else pipe.predict(X).astype("float32")

print(pd.Series(scores).describe())   # should show a spread, not a constant
preds = (scores >= THRESHOLD).astype("int8")
print("# positives @ threshold", THRESHOLD, "->", int(preds.sum()))

In [ ]:
# Keys (keep as DataFrame aligned to X)
keys_df = merged[["chr","mid","ID","strand"]].reset_index(drop=True)

# predictions_all.csv (small, OK as CSV)
pred_df = keys_df.copy()
pred_df["score"] = scores
pred_df["pred_label"] = preds
out_pred_csv = os.path.join(OUT_DIR, "predictions_all.csv")
pred_df.to_csv(out_pred_csv, index=False)
print("Saved:", out_pred_csv)

# bedGraph: use "\t" (single backslash), not "\\t"
start = (keys_df["mid"].astype(int) - 50).clip(lower=0)
end   = start + 100
bedgraph = pd.DataFrame({
    "chr":   keys_df["chr"].astype(str),
    "start": start.astype(int),
    "end":   end.astype(int),
    "score": scores.astype(float)
})
chrom_ordered = bedgraph.sort_values(["chr","start","end"], kind="mergesort").reset_index(drop=True)
bg_path = os.path.join(OUT_DIR, "predictions.bedgraph")
chrom_ordered.to_csv(bg_path, sep="\t", header=False, index=False)
print("Saved:", bg_path)


In [ ]:
import numpy as np, pandas as pd

# ---------- Load helpers (BED, FAI) ----------

def load_genes_bed(path):
    cols = ["chr","start","end","gene_id","score","strand"]
    g = pd.read_csv(path, sep="\t", header=None, names=cols, usecols=["chr","start","end","gene_id","strand"])
    g = g.sort_values(["chr","start","end"]).reset_index(drop=True)
    return g

def load_chrom_sizes(path):
    tab = pd.read_csv(path, sep="\t", header=None, usecols=[0,1], names=["chr","size"])
    tab["chr"] = tab["chr"].astype(str); tab["size"]=tab["size"].astype(int)
    return tab

# ---------- Intergenic builder & index ----------

def build_intergenic_gaps(genes_df, chrom_sizes=None):
    rows=[]
    size_map = dict(zip(chrom_sizes["chr"], chrom_sizes["size"])) if chrom_sizes is not None else {}
    for chrom, g in genes_df.groupby("chr", sort=False):
        g = g.sort_values(["start","end"]).reset_index(drop=True)
        csize = size_map.get(chrom, None)

        # head gap
        if csize is not None and len(g)>0 and g.iloc[0]["start"]>0:
            rows.append({"chr":chrom,"start":0,"end":int(g.iloc[0]["start"]),
                         "left_gene_id":np.nan,"left_strand":np.nan,
                         "right_gene_id":g.iloc[0]["gene_id"],"right_strand":g.iloc[0]["strand"]})
        # internal gaps
        for i in range(len(g)-1):
            L, R = g.iloc[i], g.iloc[i+1]
            s, e = int(L["end"]), int(R["start"])
            if e>s:
                rows.append({"chr":chrom,"start":s,"end":e,
                             "left_gene_id":L["gene_id"],"left_strand":L["strand"],
                             "right_gene_id":R["gene_id"],"right_strand":R["strand"]})
        # tail gap
        if csize is not None and len(g)>0 and int(g.iloc[-1]["end"])<csize:
            last = g.iloc[-1]
            rows.append({"chr":chrom,"start":int(last["end"]),"end":int(csize),
                         "left_gene_id":last["gene_id"],"left_strand":last["strand"],
                         "right_gene_id":np.nan,"right_strand":np.nan})
    ig = pd.DataFrame(rows)
    if ig.empty:
        ig["length"]=pd.Series(dtype="int64")
    else:
        ig["length"] = (ig["end"]-ig["start"]).astype(int)
    return ig

def index_intergenic(inter_df):
    idx={}
    for chrom, df in inter_df.groupby("chr", sort=False):
        df=df.sort_values("start").reset_index(drop=True)
        idx[chrom]={"df":df,"starts":df["start"].to_numpy(np.int64),"ends":df["end"].to_numpy(np.int64)}
    return idx

def assign_gap_point(ig_idx, chrom, pos):
    b = ig_idx.get(chrom)
    if b is None: return None
    i = np.searchsorted(b["starts"], pos, side="right") - 1
    if i>=0 and pos<b["ends"][i]: return b["df"].iloc[i]
    return None

# ---------- NEW: strand-specific 5′ rule filter ----------

def refine_tss_strand_specific(
    genes_df: pd.DataFrame,
    chrom_sizes_df: pd.DataFrame,
    tss_df: pd.DataFrame,              # needs: chr, pos, score [, strand] [, gene_id]
    score_min: float = 0.70,
    verbose: bool = True
) -> pd.DataFrame:
    """
    Keep TSS only in 'nearby intergenic' per strand rule:
      + gene: upstream of gene.start (in the gap directly before the gene)
      - gene: downstream of gene.end   (in the gap directly after the gene)
    If tss_df has 'gene_id', enforce that the target gene matches it.
    """
    if verbose: print(f"[i] Genes: {len(genes_df):,}")

    # gene coordinate maps
    gene_start = dict(zip(genes_df["gene_id"], genes_df["start"]))
    gene_end   = dict(zip(genes_df["gene_id"], genes_df["end"]))
    gene_str   = dict(zip(genes_df["gene_id"], genes_df["strand"]))

    # intergenic + index
    inter = build_intergenic_gaps(genes_df, chrom_sizes_df)
    if verbose: print(f"[i] Intergenic gaps: {len(inter):,} (mean {inter['length'].mean():.1f} bp)")
    ig_idx = index_intergenic(inter)

    # prep TSS
    keep_cols = [c for c in ["chr","pos","score","strand","gene_id"] if c in tss_df.columns]
    tss = tss_df.loc[tss_df["score"]>=score_min, keep_cols].copy()
    tss["chr"]=tss["chr"].astype(str); tss["pos"]=tss["pos"].astype(int)
    if "strand" not in tss.columns: tss["strand"] = "+"

    # assign to a gap
    rec = {k:[] for k in ["gap_start","gap_end","left_gene_id","left_strand","right_gene_id","right_strand","gap_len"]}
    keep=[]
    for _, r in tss.iterrows():
        g = assign_gap_point(ig_idx, r["chr"], int(r["pos"]))
        if g is None:
            keep.append(False)
            for k in rec: rec[k].append(np.nan)
        else:
            keep.append(True)
            rec["gap_start"].append(int(g["start"])); rec["gap_end"].append(int(g["end"]))
            rec["left_gene_id"].append(g["left_gene_id"]);   rec["left_strand"].append(g["left_strand"])
            rec["right_gene_id"].append(g["right_gene_id"]); rec["right_strand"].append(g["right_strand"])
            rec["gap_len"].append(int(g["length"]))
    tss = tss.reset_index(drop=True)
    for k,v in rec.items(): tss[k]=v
    tss = tss.loc[keep].copy()
    if verbose: print(f"[i] TSS in gaps: {len(tss):,}")

    # Pull the neighbor gene TSS/end
    tss["right_start"] = tss["right_gene_id"].map(gene_start)
    tss["left_end"]    = tss["left_gene_id"].map(gene_end)

    # STRAND-SPECIFIC ACCEPTANCE:
    # + gene -> use the right gene in the gap; candidate must be strictly upstream of its start
    # - gene -> use the left  gene in the gap; candidate must be strictly downstream of its end
    keep_plus  = (tss["right_gene_id"].notna()) & (tss["right_strand"]=="+") & (tss["pos"] <  tss["right_start"])
    keep_minus = (tss["left_gene_id"].notna())  & (tss["left_strand"]=="-")  & (tss["pos"] >  tss["left_end"])
    tss = tss.loc[keep_plus | keep_minus].copy()
    if verbose: print(f"[i] After strand rule (+ upstream / - downstream): {len(tss):,}")

    # Define the *target* gene implied by the rule
    tss["target_gene"]   = np.where(keep_plus,  tss["right_gene_id"], tss["left_gene_id"])
    tss["target_strand"] = np.where(keep_plus,  tss["right_strand"],  tss["left_strand"])
    tss["target_tss"]    = np.where(keep_plus,  tss["right_start"],   tss["left_end"])  # TSS coord used for tie-break logic

    # If a gene_id is present in the TSS table, enforce it equals the target
    if "gene_id" in tss.columns and tss["gene_id"].notna().any():
        before = len(tss)
        tss = tss.loc[tss["gene_id"] == tss["target_gene"]].copy()
        if verbose: print(f"[i] Enforcing provided gene_id match: {before} -> {len(tss)}")

    # Tie-break: highest score, then farthest upstream relative to the target gene strand
    def upstream_key(row):
        if row["target_strand"] == "-":
            # upstream of a -strand gene is toward larger genomic coords
            return -int(row["pos"])  # sort asc -> larger pos first
        else:
            # upstream of a +strand gene is toward smaller coords
            return int(row["pos"])

    tss["__upkey"] = tss.apply(upstream_key, axis=1)
    tss = tss.sort_values(["target_gene","score","__upkey"], ascending=[True, False, True]).copy()
    final = tss.groupby("target_gene", as_index=False).head(1).drop(columns="__upkey")

    # Output columns
    cols = [
        "chr","pos","strand","score",
        "gap_start","gap_end","gap_len",
        "left_gene_id","left_strand","right_gene_id","right_strand",
        "target_gene","target_strand"
    ]
    return final[cols].copy()


In [ ]:
# Paths (edit)
GENES_BED = "/rhome/zli529/lab/LncRNA_chip_prediction/Final/window1000_bin100/prediction/results_intergenic_TSS/PlasmoDB-48_Pfalciparum3D7_Genes.bed"
FAI_PATH  = "/rhome/zli529/lab/PlasmoDB_Genome/PlasmoDB_v48/PlasmoDB-48_Pfalciparum3D7_Genome.fasta.fai"
TSS_IN    = "/rhome/zli529/lab/LncRNA_chip_prediction/Final/window1000_bin100/prediction/results_intergenic_TSS/predictions_all_fixed.tsv"   # chr, pos, score [, strand] [, gene_id]
OUT_TSV   = "/rhome/zli529/lab/LncRNA_chip_prediction/Final/window1000_bin100/prediction/results_intergenic_TSS/refined_strand_rule.tsv"
OUT_BED   = "/rhome/zli529/lab/LncRNA_chip_prediction/Final/window1000_bin100/prediction/results_intergenic_TSS/refined_strand_rule.bed"

# Load inputs
genes = load_genes_bed(GENES_BED)
chrom_sizes = load_chrom_sizes(FAI_PATH)
tss = pd.read_csv(TSS_IN, sep="\t")

# Run filter
final = refine_tss_strand_specific(genes, chrom_sizes, tss, score_min=0.70, verbose=True)

# Save
final.to_csv(OUT_TSV, sep="\t", index=False)
(final[["chr","pos","pos","target_gene","score","strand"]]
 .to_csv(OUT_BED, sep="\t", header=False, index=False))
print("Saved:", OUT_TSV, "\nSaved:", OUT_BED)
